In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("readcsvfile").getOrCreate()
df1 = spark.range(1000000)
df2 = spark.range(1000000)

In [4]:
joindf = df1.join(df2, df1.id==df2.id)
joindf.show()

+---+---+
| id| id|
+---+---+
|  0|  0|
|  1|  1|
|  2|  2|
|  3|  3|
|  4|  4|
|  5|  5|
|  6|  6|
|  7|  7|
|  8|  8|
|  9|  9|
| 10| 10|
| 11| 11|
| 12| 12|
| 13| 13|
| 14| 14|
| 15| 15|
| 16| 16|
| 17| 17|
| 18| 18|
| 19| 19|
+---+---+
only showing top 20 rows



In [5]:
joindf.explain(True)

== Parsed Logical Plan ==
Join Inner, (id#4L = id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- Range (0, 1000000, step=1, splits=Some(16))

== Analyzed Logical Plan ==
id: bigint, id: bigint
Join Inner, (id#4L = id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- Range (0, 1000000, step=1, splits=Some(16))

== Optimized Logical Plan ==
Join Inner, (id#4L = id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- Range (0, 1000000, step=1, splits=Some(16))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#4L], [id#6L], Inner, BuildRight, false
   :- Range (0, 1000000, step=1, splits=16)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=62]
      +- Range (0, 1000000, step=1, splits=16)



In [6]:
from pyspark.sql.functions import *

In [7]:
joindf2 = df1.join(df2.hint("broadcast"),df1.id==df2.id)
joindf2.show()

+---+---+
| id| id|
+---+---+
|  0|  0|
|  1|  1|
|  2|  2|
|  3|  3|
|  4|  4|
|  5|  5|
|  6|  6|
|  7|  7|
|  8|  8|
|  9|  9|
| 10| 10|
| 11| 11|
| 12| 12|
| 13| 13|
| 14| 14|
| 15| 15|
| 16| 16|
| 17| 17|
| 18| 18|
| 19| 19|
+---+---+
only showing top 20 rows



In [9]:
joindf2.explain(True)

== Parsed Logical Plan ==
Join Inner, (id#4L = id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- ResolvedHint (strategy=broadcast)
   +- Range (0, 1000000, step=1, splits=Some(16))

== Analyzed Logical Plan ==
id: bigint, id: bigint
Join Inner, (id#4L = id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- ResolvedHint (strategy=broadcast)
   +- Range (0, 1000000, step=1, splits=Some(16))

== Optimized Logical Plan ==
Join Inner, (id#4L = id#6L), rightHint=(strategy=broadcast)
:- Range (0, 1000000, step=1, splits=Some(16))
+- Range (0, 1000000, step=1, splits=Some(16))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#4L], [id#6L], Inner, BuildRight, false
   :- Range (0, 1000000, step=1, splits=16)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=176]
      +- Range (0, 1000000, step=1, splits=16)



In [14]:
joindf3 = df1.join(df2.hint("merge"),df1.id==df2.id)
joindf3.show()

+---+---+
| id| id|
+---+---+
|  0|  0|
|  7|  7|
| 19| 19|
| 22| 22|
| 26| 26|
| 29| 29|
| 34| 34|
| 50| 50|
| 54| 54|
| 65| 65|
| 77| 77|
| 94| 94|
|112|112|
|113|113|
|126|126|
|130|130|
|149|149|
|155|155|
|167|167|
|184|184|
+---+---+
only showing top 20 rows



In [15]:
joindf3.explain(True)

== Parsed Logical Plan ==
Join Inner, (id#4L = id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- ResolvedHint (strategy=merge)
   +- Range (0, 1000000, step=1, splits=Some(16))

== Analyzed Logical Plan ==
id: bigint, id: bigint
Join Inner, (id#4L = id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- ResolvedHint (strategy=merge)
   +- Range (0, 1000000, step=1, splits=Some(16))

== Optimized Logical Plan ==
Join Inner, (id#4L = id#6L), rightHint=(strategy=merge)
:- Range (0, 1000000, step=1, splits=Some(16))
+- Range (0, 1000000, step=1, splits=Some(16))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [id#4L], [id#6L], Inner
   :- Sort [id#4L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(id#4L, 200), ENSURE_REQUIREMENTS, [plan_id=449]
   :     +- Range (0, 1000000, step=1, splits=16)
   +- Sort [id#6L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(id#6L, 200), ENSURE_REQUIREMENTS, [plan_id=450]
         +- Range (0, 

In [19]:
joindf4 = df1.join(df2.hint("SHUFFLE_REPLICATE_NL"),df1.id>=df2.id)
joindf4.show()

+---+---+
| id| id|
+---+---+
|  0|  0|
|  1|  0|
|  1|  1|
|  2|  0|
|  2|  1|
|  2|  2|
|  3|  0|
|  3|  1|
|  3|  2|
|  3|  3|
|  4|  0|
|  4|  1|
|  4|  2|
|  4|  3|
|  4|  4|
|  5|  0|
|  5|  1|
|  5|  2|
|  5|  3|
|  5|  4|
+---+---+
only showing top 20 rows



In [20]:
joindf4.explain(True)

== Parsed Logical Plan ==
Join Inner, (id#4L >= id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- ResolvedHint (strategy=shuffle_replicate_nl)
   +- Range (0, 1000000, step=1, splits=Some(16))

== Analyzed Logical Plan ==
id: bigint, id: bigint
Join Inner, (id#4L >= id#6L)
:- Range (0, 1000000, step=1, splits=Some(16))
+- ResolvedHint (strategy=shuffle_replicate_nl)
   +- Range (0, 1000000, step=1, splits=Some(16))

== Optimized Logical Plan ==
Join Inner, (id#4L >= id#6L), rightHint=(strategy=shuffle_replicate_nl)
:- Range (0, 1000000, step=1, splits=Some(16))
+- Range (0, 1000000, step=1, splits=Some(16))

== Physical Plan ==
CartesianProduct (id#4L >= id#6L)
:- *(1) Range (0, 1000000, step=1, splits=16)
+- *(2) Range (0, 1000000, step=1, splits=16)

